# 12: Does error decorrelation show value in the severity-weighted currency? (EXPLORATORY)

Every decorrelation test so far used the wrong units. Overall rho counted a false positive the
same as a missed PDR. The severe-case test lumped grades 3 and 4. If decorrelation matters, it
should matter when **dependence is measured in severity-weighted terms** and tested against the
**severity-weighted advantage**.

All predictors are measured on **validation**, the outcome on the **fresh set**, so every test is
out of sample. Weights 500 / 200 / 15 for grade 4 / 3 / 2 misses, 1 per false positive.

1. **Cost-weighted error correlation.** Correlation between the two models' *per-case cost
   contributions* (a missed PDR contributes 500, a false positive 1), rather than between binary
   error flags. This is rho in the currency the paper actually uses.
2. **PDR-specific dependence.** Joint grade-4 miss rate versus the independence benchmark. Do
   models fail together on proliferative disease specifically?
3. **Shared blind-spot cost.** Weighted cost of the cases both models miss, normalised.
4. **Decomposition in weighted currency**, overall and for grade 4 alone: how much of the
   weighted advantage is explained by marginal performance under independence, and how much the
   observed dependence adds or subtracts.
5. Head to head against the predictors that already failed, so the comparison is explicit.

CPU runtime, about 20 minutes. Run cells 1-5.

In [1]:
# Cell 1: load validation and fresh predictions
import io, os, json, itertools, datetime, time
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc=gbuild('drive','v3')
F_PRED='1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'; F_RES='1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'; F_FIG='18eyHRWuqFcUrYiL8nPtpIalV7CreLkMK'
def dl(fid,p):
    with io.FileIO(p,'wb') as fh:
        d=MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done=False
        while not done: _,done=d.next_chunk()
def find_id(n,f):
    r=svc.files().list(q=f"'{f}' in parents and name='{n}' and trashed=false", fields='files(id)').execute().get('files',[])
    return r[0]['id'] if r else None
def up(p,f,m):
    svc.files().create(body={'name':os.path.basename(p),'parents':[f]}, media_body=MediaFileUpload(p,mimetype=m), fields='id').execute()
    print('  uploaded', os.path.basename(p))

ARCHS=['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
       'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
       'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS=[0,1,2]; PAIRS=list(itertools.combinations(ARCHS,2)); COMBOS=[(i,j) for i in SEEDS for j in SEEDS]
def tag(k): return '' if k==0 else f'_seed{k}'
BURDEN=0.25; W={4:500,3:200,2:15}; WFP=1

def load(prefix, split):
    P, ref = {}, None
    for a in ARCHS:
        for k in SEEDS:
            fn=f'{prefix}{a}{tag(k)}_v2.csv'
            if not os.path.exists(fn): dl(find_id(fn,F_PRED), fn)
            d=pd.read_csv(fn); assert set(d['split'])=={split}
            if ref is None: ref=d[['image_id','patient_id','true_grade','true_label']].reset_index(drop=True)
            P[(a,k)]=d['predicted_probability'].to_numpy(np.float64)
    return P, ref
PV,RV=load('eyepacs_predictions_val_','val'); PF,RF=load('eyepacs_predictions_fresh_','fresh')
GV=RV['true_grade'].to_numpy(int); LV=RV['true_label'].to_numpy(int)
GF=RF['true_grade'].to_numpy(int); LF=RF['true_label'].to_numpy(int)
print(f'validation {len(GV)} | fresh {len(GF)} (grade4 {(GF==4).sum()}, grade3 {(GF==3).sum()})')

validation 2900 | fresh 15794 (grade4 328, grade3 384)


In [2]:
# Cell 2: per-case cost contributions, and the weighted dependence quantities
def setup(G, L):
    n=len(G); R=int(round(BURDEN*n))
    missw=np.where(L==1, np.where(G==4,W[4],np.where(G==3,W[3],W[2])), 0.0)
    return n, R, missw, (L==0).astype(float)

def topR(p,R):
    m=np.zeros(len(p),bool); m[np.argsort(-p,kind='stable')[:R]]=True; return m

def per_model(P, G, L):
    n,R,missw,neg = setup(G,L)
    REF={k:topR(v,R) for k,v in P.items()}
    COST={k: missw*(~REF[k]) + WFP*neg*REF[k] for k in P}      # per-case cost contribution
    RK={k:rankdata(v)/n for k,v in P.items()}
    return n,R,missw,neg,REF,COST,RK

def quantities(P, G, L):
    n,R,missw,neg,REF,COST,RK = per_model(P,G,L)
    g4=(G==4); n4=g4.sum(); tw=missw.sum()
    rows=[]
    for a,b in PAIRS:
        acc={'rho_cost':[], 'rho_binary':[], 'pdr_dep_excess':[], 'shared_blindspot_cost':[],
             'wadv_matched':[], 'wadv_indep_pred':[], 'pdr_avoided':[]}
        for i,j in COMBOS:
            ca, cb = COST[(a,i)], COST[(b,j)]
            acc['rho_cost'].append(float(np.corrcoef(ca, cb)[0,1]))
            ea=(~REF[(a,i)])&(L==1) | (REF[(a,i)]&(L==0)); eb=(~REF[(b,j)])&(L==1) | (REF[(b,j)]&(L==0))
            acc['rho_binary'].append(float(np.corrcoef(ea.astype(float), eb.astype(float))[0,1]))
            ma4=(~REF[(a,i)])&g4; mb4=(~REF[(b,j)])&g4
            p4a, p4b = ma4.sum()/n4, mb4.sum()/n4
            acc['pdr_dep_excess'].append(float((ma4&mb4).sum()/n4 - p4a*p4b))
            both=(~REF[(a,i)])&(~REF[(b,j)])&(L==1)
            acc['shared_blindspot_cost'].append(float(missw[both].sum()/tw))
            # weighted advantage at matched workload, and what independence alone would predict
            orf=topR(np.maximum(RK[(a,i)],RK[(b,j)]),R)
            wc=lambda m: float(missw[(~m)&(L==1)].sum() + WFP*((m)&(L==0)).sum())
            base=min(wc(REF[(a,i)]), wc(REF[(b,j)]))
            acc['wadv_matched'].append(base - wc(orf))
            # independence benchmark: expected shared weighted miss if errors were independent
            pa=(~REF[(a,i)])&(L==1); pb=(~REF[(b,j)])&(L==1)
            shared_indep=float((missw*(pa.mean()*pb.mean())).sum())
            acc['wadv_indep_pred'].append(base - shared_indep)
            acc['pdr_avoided'].append(float(min(ma4.sum(), mb4.sum()) - ((~orf)&g4).sum()))
        rows.append({'model_a':a,'model_b':b, **{k:float(np.mean(v)) for k,v in acc.items()}})
    return pd.DataFrame(rows)

t0=time.time(); QV=quantities(PV,GV,LV); QF=quantities(PF,GF,LF)
print(f'computed in {time.time()-t0:.0f}s')
print(QF[['rho_cost','rho_binary','pdr_dep_excess','shared_blindspot_cost','wadv_matched','pdr_avoided']]
      .describe().round(4).to_string())

computed in 4s
       rho_cost  rho_binary  pdr_dep_excess  shared_blindspot_cost  wadv_matched  pdr_avoided
count   55.0000     55.0000         55.0000                55.0000       55.0000      55.0000
mean     0.4354      0.4495          0.0173                 0.0537     2659.4768       3.0141
std      0.1077      0.0938          0.0060                 0.0070     1245.7617       1.6062
min      0.2978      0.3932          0.0092                 0.0446      549.5556      -0.5556
25%      0.3696      0.4045          0.0124                 0.0482     1605.1667       1.7778
50%      0.4060      0.4234          0.0151                 0.0515     2732.2222       3.0000
75%      0.4668      0.4346          0.0228                 0.0583     3698.6667       3.9444
max      0.8034      0.8055          0.0315                 0.0759     4792.3333       6.1111


In [3]:
# Cell 3: does decorrelation predict the severity-weighted advantage? (out of sample)
key=lambda df: df.set_index(['model_a','model_b'])
gv, gf = key(QV), key(QF)
idx=[(a,b) for a,b in PAIRS]
Y_W  = np.array([gf.loc[p,'wadv_matched'] for p in idx])     # severity-weighted advantage, fresh
Y_P4 = np.array([gf.loc[p,'pdr_avoided']  for p in idx])     # PDR misses avoided, fresh
CANDS={
 'cost-weighted error correlation':      np.array([gv.loc[p,'rho_cost'] for p in idx]),
 'binary error correlation (old rho)':   np.array([gv.loc[p,'rho_binary'] for p in idx]),
 'PDR-specific dependence excess':       np.array([gv.loc[p,'pdr_dep_excess'] for p in idx]),
 'shared blind-spot cost':               np.array([gv.loc[p,'shared_blindspot_cost'] for p in idx]),
}
rng=np.random.default_rng(20260922)
def mlci(x,y,B=2000):
    out=[]
    for _ in range(B):
        s=rng.choice(ARCHS,len(ARCHS),replace=True)
        ps=[(u,v) if (u,v) in gf.index else (v,u) for u,v in itertools.combinations(s,2) if u!=v]
        if len(set(ps))<5: continue
        k=[idx.index(p) for p in ps]
        out.append(spearmanr(x[k],y[k]).correlation)
    return [round(float(v),3) for v in np.nanpercentile(out,[2.5,97.5])]
rows=[]
for name,x in CANDS.items():
    rows.append({'predictor (validation)':name,
                 'vs weighted advantage':round(float(spearmanr(x,Y_W).correlation),3),
                 'ci (weighted)':mlci(x,Y_W),
                 'vs PDR misses avoided':round(float(spearmanr(x,Y_P4).correlation),3),
                 'ci (PDR)':mlci(x,Y_P4)})
PRED=pd.DataFrame(rows); print(PRED.to_string(index=False))
print('\nNegative correlation = less dependence goes with more benefit (the hypothesis).')

            predictor (validation)  vs weighted advantage   ci (weighted)  vs PDR misses avoided        ci (PDR)
   cost-weighted error correlation                  0.179 [-0.391, 0.597]                  0.012  [-0.47, 0.466]
binary error correlation (old rho)                  0.060 [-0.551, 0.488]                 -0.003 [-0.567, 0.474]
    PDR-specific dependence excess                  0.248 [-0.324, 0.627]                  0.107 [-0.427, 0.531]
            shared blind-spot cost                  0.180 [-0.414, 0.576]                  0.030 [-0.514, 0.447]

Negative correlation = less dependence goes with more benefit (the hypothesis).


In [4]:
# Cell 4: decomposition in weighted currency, overall and PDR only
obs_w = float(gf['wadv_matched'].mean()); ind_w = float(gf['wadv_indep_pred'].mean())
n4=int((GF==4).sum())
DEC={'mean_weighted_advantage_observed':round(obs_w,1),
     'mean_weighted_advantage_if_errors_independent':round(ind_w,1),
     'added_by_observed_dependence':round(obs_w-ind_w,1),
     'share_from_dependence':round((obs_w-ind_w)/obs_w,3) if obs_w else None,
     'mean_shared_blindspot_cost_share':round(float(gf['shared_blindspot_cost'].mean()),4),
     'pdr_dependence_excess_mean':round(float(gf['pdr_dep_excess'].mean()),5),
     'pairs_with_positive_pdr_dependence_of_55':int((gf['pdr_dep_excess']>0).sum()),
     'mean_pdr_misses_avoided':round(float(gf['pdr_avoided'].mean()),2)}
print(json.dumps(DEC, indent=2))
print('\nPositive pdr_dependence_excess means models miss the SAME proliferative eyes more often')
print('than chance. If that holds for most pairs, shared blind spots cap the PDR benefit, and')
print('decorrelation is not what produces it.')

{
  "mean_weighted_advantage_observed": 2659.5,
  "mean_weighted_advantage_if_errors_independent": 24531.0,
  "added_by_observed_dependence": -21871.5,
  "share_from_dependence": -8.224,
  "mean_shared_blindspot_cost_share": 0.0537,
  "pdr_dependence_excess_mean": 0.01731,
  "pairs_with_positive_pdr_dependence_of_55": 55,
  "mean_pdr_misses_avoided": 3.01
}

Positive pdr_dependence_excess means models miss the SAME proliferative eyes more often
than chance. If that holds for most pairs, shared blind spots cap the PDR benefit, and
decorrelation is not what produces it.


In [5]:
# Cell 5: figure and save
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,3,figsize=(15,4.4))
ax[0].scatter(CANDS['cost-weighted error correlation'], Y_W, s=18, color='C0')
ax[0].set_xlabel('validation cost-weighted error correlation'); ax[0].set_ylabel('fresh weighted advantage')
ax[0].set_title('rho in the severity currency')
ax[1].scatter(CANDS['PDR-specific dependence excess'], Y_P4, s=18, color='C3')
ax[1].set_xlabel('validation PDR dependence excess'); ax[1].set_ylabel('fresh PDR misses avoided')
ax[1].set_title('PDR-specific dependence')
ax[2].scatter(CANDS['binary error correlation (old rho)'], Y_W, s=18, color='grey')
ax[2].set_xlabel('validation binary error correlation'); ax[2].set_ylabel('fresh weighted advantage')
ax[2].set_title('the predictor that failed')
for a_ in ax: a_.axhline(0,color='grey',lw=0.8)
fig.suptitle('Does decorrelation predict benefit once severity is baked in? (exploratory, out of sample)')
fig.tight_layout(); fig.savefig('decorrelation_weighted_v3.png', dpi=200)

OUT={'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
     'status':'EXPLORATORY; out-of-sample (predictors from validation, outcomes from fresh set)',
     'weights':W,'false_positive_weight':WFP,'burden':BURDEN,
     'predictors':PRED.to_dict(orient='records'),'decomposition':DEC}
with open('decorrelation_weighted_v3.json','w') as f: json.dump(OUT,f,indent=2,default=str)
QF.to_csv('weighted_dependence_fresh_v3.csv',index=False); QV.to_csv('weighted_dependence_val_v3.csv',index=False)
for fn,m in (('decorrelation_weighted_v3.json','application/json'),
             ('weighted_dependence_fresh_v3.csv','text/csv'),('weighted_dependence_val_v3.csv','text/csv')):
    up(fn,F_RES,m)
up('decorrelation_weighted_v3.png',F_FIG,'image/png')
print('saved')

  uploaded decorrelation_weighted_v3.json
  uploaded weighted_dependence_fresh_v3.csv
  uploaded weighted_dependence_val_v3.csv
  uploaded decorrelation_weighted_v3.png
saved
